# Cross-check: Butler visitTable vs constDb visitTable

**Purpose:**  
Verify the equivalence of visit numbers, tracts and patches between two independent Rubin/LSST data sources:
- `visitTable` from the **Butler main** repository  (from https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/1-correlate-dp2-with-fink-alerts/notebooks/2026-03-25_FindObservationsInButlerRegistryInTractPatch.ipynb)
- `visitTable` from the **consDb** (consolidated database) (from https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_DP2_ConstDB_Butler_LSSTCam_VisitsTractPatch.ipynb)

This notebook also provides utilities to map diaObject alerts (from Fink lightcurves) to their tract/patch,
and to generate the `dataQuery` string for a `bps submit` DRP pipeline job.

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Date:** 2026-03-28


## 0. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR = "data_fromlsst"

# Butler main visitTable (with tracts/patches)
FILE_BUTLER = os.path.join(DATA_DIR, "visitTable-2025041500138-2026031900284_N52726_WithTracts.parquet")

# consDb visitTable (with tracts/patches)
FILE_CONSDB = os.path.join(
    DATA_DIR, "constDbVisitTable-2025041500043-2026032500675_N85366_WithTracts.parquet"
)

print(f"Butler file : {FILE_BUTLER}")
print(f"consDb file : {FILE_CONSDB}")
print(f"Butler data exists: {os.path.exists(FILE_BUTLER)}")
print(f"consDb data exists: {os.path.exists(FILE_CONSDB)}")

## 1. Load the two tables

In [ ]:
df_butler = pd.read_parquet(FILE_BUTLER)
df_consdb = pd.read_parquet(FILE_CONSDB)

print("=" * 60)
print(f"Butler visitTable  →  {df_butler.shape[0]:,} rows  ×  {df_butler.shape[1]} columns")
print(f"consDb visitTable  →  {df_consdb.shape[0]:,} rows  ×  {df_consdb.shape[1]} columns")

### 1.1 Schema inspection

In [ ]:
print("── Butler columns ──")
print(df_butler.dtypes)
print()
print("── consDb columns ──")
print(df_consdb.dtypes)

In [ ]:
print("── Butler head ──")
display(df_butler.head(3))
print("── consDb head ──")
display(df_consdb.head(3))

### 1.2 Identify the key columns

Adjust these names if the actual column names in the parquet differ.

In [ ]:
# ── COLUMN NAME CONFIGURATION ─────────────────────────────────────────────────
# Adapt to the real column names found above.

# Visit identifier (13-digit Butler visitId or equivalent)
# COL_VISIT_BUTLER = "visitId"     # e.g. 'visitId', 'visit', 'r:visit'
COL_VISIT_BUTLER = "id"  # e.g. 'visitId', 'visit', 'r:visit'
COL_VISIT_CONSDB = "visit_id"  # e.g. 'visit_id', 'visitId'

# Sky-map geometry columns
COL_TRACT_BUTLER = "tract"
COL_PATCH_BUTLER = "patch"
COL_TRACT_CONSDB = "tract"
COL_PATCH_CONSDB = "patch"

# Optional: detector, band/filter
COL_DETECTOR_BUTLER = "detector"  # set to None if absent
COL_DETECTOR_CONSDB = "detector"
COL_BAND_BUTLER = "band"  # set to None if absent
COL_BAND_CONSDB = "band"


# Auto-detect: override config above with whatever is actually in the DataFrames
def detect_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            print(f"  [{label}] found column: '{c}'")
            return c
    print(f"  [{label}] WARNING – none of {candidates} found in columns")
    return None


print("Auto-detecting visit column …")
COL_VISIT_BUTLER = detect_col(df_butler, ["visitId", "visit", "r:visit", "visit_id", "id"], "butler")
COL_VISIT_CONSDB = detect_col(df_consdb, ["visit_id", "visitId", "visit", "r:visit", "id"], "consdb")

print("Auto-detecting tract/patch …")
COL_TRACT_BUTLER = detect_col(df_butler, ["tract", "skymap_tract"], "butler")
COL_PATCH_BUTLER = detect_col(df_butler, ["patch", "skymap_patch"], "butler")
COL_TRACT_CONSDB = detect_col(df_consdb, ["tract", "skymap_tract"], "consdb")
COL_PATCH_CONSDB = detect_col(df_consdb, ["patch", "skymap_patch"], "consdb")

print("Auto-detecting band …")
COL_BAND_BUTLER = detect_col(df_butler, ["band", "filter", "physical_filter"], "butler")
COL_BAND_CONSDB = detect_col(df_consdb, ["band", "filter", "physical_filter"], "consdb")

## 2. Visit-set comparison

In [ ]:
visits_butler = set(df_butler[COL_VISIT_BUTLER].dropna().astype(int))
visits_consdb = set(df_consdb[COL_VISIT_CONSDB].dropna().astype(int))

common = visits_butler & visits_consdb
only_butler = visits_butler - visits_consdb
only_consdb = visits_consdb - visits_butler

print(f"Visits in Butler only   : {len(visits_butler):,}")
print(f"Visits in consDb only   : {len(visits_consdb):,}")
print(f"Visits in BOTH          : {len(common):,}")
print(f"Only in Butler (not consDb): {len(only_butler):,}")
print(f"Only in consDb (not Butler): {len(only_consdb):,}")
print()
overlap_pct = 100 * len(common) / max(len(visits_butler), len(visits_consdb))
print(f"Overlap fraction (vs larger set): {overlap_pct:.1f} %")

In [ ]:
# Venn-style bar chart
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Butler only", "Common", "consDb only"]
values = [len(only_butler), len(common), len(only_consdb)]
colors = ["#4C9BE8", "#2ECC71", "#E8754C"]
bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=1.4)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontsize=11,
    )
ax.set_ylabel("Number of visits")
ax.set_title("Visit overlap: Butler visitTable vs consDb visitTable")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 3. Tract / Patch consistency on common visits

For visits present in both tables, compare the (tract, patch) assignments.

In [ ]:
# Build merged table on common visits
cols_butler = [COL_VISIT_BUTLER, COL_TRACT_BUTLER, COL_PATCH_BUTLER]
cols_consdb = [COL_VISIT_CONSDB, COL_TRACT_CONSDB, COL_PATCH_CONSDB]
if COL_BAND_BUTLER:
    cols_butler.append(COL_BAND_BUTLER)
if COL_BAND_CONSDB:
    cols_consdb.append(COL_BAND_CONSDB)

sub_b = (
    df_butler[cols_butler]
    .copy()
    .rename(
        columns={
            COL_VISIT_BUTLER: "visitId",
            COL_TRACT_BUTLER: "tract_butler",
            COL_PATCH_BUTLER: "patch_butler",
        }
    )
)
if COL_BAND_BUTLER:
    sub_b = sub_b.rename(columns={COL_BAND_BUTLER: "band_butler"})

sub_c = (
    df_consdb[cols_consdb]
    .copy()
    .rename(
        columns={
            COL_VISIT_CONSDB: "visitId",
            COL_TRACT_CONSDB: "tract_consdb",
            COL_PATCH_CONSDB: "patch_consdb",
        }
    )
)
if COL_BAND_CONSDB:
    sub_c = sub_c.rename(columns={COL_BAND_CONSDB: "band_consdb"})

sub_b["visitId"] = sub_b["visitId"].astype(int)
sub_c["visitId"] = sub_c["visitId"].astype(int)

merged = sub_b.merge(sub_c, on="visitId", how="inner")
print(f"Merged table shape: {merged.shape}")
display(merged.head(5))

In [ ]:
# Tract agreement
merged["tract_match"] = merged["tract_butler"] == merged["tract_consdb"]
merged["patch_match"] = merged["patch_butler"].astype(str) == merged["patch_consdb"].astype(str)
merged["full_match"] = merged["tract_match"] & merged["patch_match"]

n_total = len(merged)
n_tract_ok = merged["tract_match"].sum()
n_patch_ok = merged["patch_match"].sum()
n_full_ok = merged["full_match"].sum()

print(f"Common visits (rows): {n_total:,}")
print(f"Tract matches  : {n_tract_ok:,} / {n_total:,}  ({100 * n_tract_ok / n_total:.2f} %)")
print(f"Patch matches  : {n_patch_ok:,} / {n_total:,}  ({100 * n_patch_ok / n_total:.2f} %)")
print(f"Full match     : {n_full_ok:,} / {n_total:,}  ({100 * n_full_ok / n_total:.2f} %)")

In [ ]:
# Show mismatches if any
mismatches = merged[~merged["full_match"]]
print(f"Mismatched rows: {len(mismatches)}")
if len(mismatches) > 0:
    display(mismatches.head(20))

### 3.1 Distribution of tracts and patches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tract distribution
ax = axes[0]
merged["tract_butler"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C9BE8")
ax.set_title("Tract distribution (Butler, common visits)")
ax.set_xlabel("tract")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)

# Patch distribution
ax = axes[1]
merged["patch_butler"].astype(str).value_counts().sort_index().plot(kind="bar", ax=ax, color="#E8754C")
ax.set_title("Patch distribution (Butler, common visits)")
ax.set_xlabel("patch")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 2D histogram: tract vs patch
tract_vals = merged["tract_butler"].astype(int)
patch_vals = merged["patch_butler"].astype(str)  # patch may be '0,0' style string

# If patch is numeric, plot 2D
try:
    patch_num = merged["patch_butler"].astype(int)
    fig, ax = plt.subplots(figsize=(9, 6))
    h = ax.hist2d(tract_vals, patch_num, bins=50, norm=LogNorm(), cmap="plasma")
    plt.colorbar(h[3], ax=ax, label="count (log scale)")
    ax.set_xlabel("tract")
    ax.set_ylabel("patch")
    ax.set_title("2D distribution: tract vs patch (Butler, common visits)")
    plt.tight_layout()
    plt.show()
except (ValueError, TypeError):
    print("Patch column is not purely numeric – skipping 2D histogram.")
    print("Unique patch values sample:", sorted(set(patch_vals.tolist()))[:20])

## 4. Build a canonical lookup table: visitId → (tract, patch)

This will be used later to map diaObject alerts to their sky-map position.

In [ ]:
# Prefer Butler as reference; fallback to consDb for visits absent from Butler
lookup_butler = sub_b.rename(columns={"tract_butler": "tract", "patch_butler": "patch"})
lookup_consdb = sub_c.rename(columns={"tract_consdb": "tract", "patch_consdb": "patch"})

lookup_all = pd.concat([lookup_butler, lookup_consdb], ignore_index=True)
lookup_all = lookup_all.drop_duplicates(subset=["visitId"]).set_index("visitId")

print(f"Canonical lookup table: {len(lookup_all):,} unique visitIds")
display(lookup_all.head(5))

## 5. Map diaObject alerts → tract / patch

This section loads a lightcurve from a Fink alert notebook output and cross-matches each alert's visitId against the canonical lookup.

In [ ]:
# ── CONFIGURATION: set the diaObject and path to its alert data ───────────────
# Replace with the actual diaObject ID and data file from your Fink notebooks.

DIAOBJECT_ID = None  # e.g. 12345678901234   ← set this
ALERT_FILE = None  # e.g. "../data/alerts_diaobj_12345.parquet"  ← set this

# Column names in the alert dataframe
COL_ALERT_VISIT = "r:visit"  # 13-digit Butler visitId in Fink alerts
COL_ALERT_DETECTOR = "r:detector"  # detector number (optional)
COL_ALERT_BAND = "i:fid"  # filter id (1=g, 2=r, 3=i for ZTF-style; adjust for LSST)
COL_ALERT_DIAOBJ = "i:diaObjectId"

print("Configure DIAOBJECT_ID and ALERT_FILE above, then re-run this cell.")

In [ ]:
if ALERT_FILE is not None and os.path.exists(str(ALERT_FILE)):
    df_alerts = pd.read_parquet(ALERT_FILE)
    print(f"Alerts loaded: {df_alerts.shape}")
    display(df_alerts.head(3))
else:
    print("[SKIP] ALERT_FILE not set or not found – using a synthetic example.")
    # Synthetic placeholder using visitIds drawn from the Butler table
    rng = np.random.default_rng(42)
    sample_visits = rng.choice(list(visits_butler), size=min(20, len(visits_butler)), replace=False)
    df_alerts = pd.DataFrame(
        {
            COL_ALERT_VISIT: sample_visits,
            COL_ALERT_DETECTOR: rng.integers(0, 189, size=len(sample_visits)),
            COL_ALERT_BAND: rng.choice([1, 2, 3], size=len(sample_visits)),
            COL_ALERT_DIAOBJ: DIAOBJECT_ID if DIAOBJECT_ID else 0,
        }
    )
    print(f"Synthetic alert table created: {df_alerts.shape}")
    display(df_alerts.head())

In [ ]:
# Cross-match alerts to lookup table
df_alerts["visitId_int"] = df_alerts[COL_ALERT_VISIT].astype(int)

df_matched = df_alerts.join(lookup_all[["tract", "patch"]], on="visitId_int", how="left")

n_matched = df_matched["tract"].notna().sum()
n_unmatched = df_matched["tract"].isna().sum()
print(f"Alerts matched to tract/patch : {n_matched} / {len(df_alerts)}")
print(f"Unmatched (visitId not in lookup): {n_unmatched}")
display(df_matched.head(10))

In [ ]:
# Check that all alerts belong to the SAME tract and patch (as expected for a point source)
unique_tracts = df_matched["tract"].dropna().unique()
unique_patches = df_matched["patch"].dropna().astype(str).unique()

print(f"Unique tracts  : {unique_tracts}")
print(f"Unique patches : {unique_patches}")

if len(unique_tracts) == 1 and len(unique_patches) == 1:
    print()
    print("✅  All alerts belong to a SINGLE tract/patch — consistent sky position.")
    TRACT = int(unique_tracts[0])
    PATCH = str(unique_patches[0])
    print(f"   tract = {TRACT}")
    print(f"   patch = {PATCH}")
else:
    print()
    print("⚠️  Alerts span multiple tracts or patches — investigate further.")
    TRACT, PATCH = None, None

## 6. Generate the `dataQuery` string for `bps submit`

The DRP pipeline is launched via `bps submit <yaml>` with a `dataQuery` that selects the relevant
visits and sky-map position.

- see my example to run the DRP myself here : https://github.com/sylvielsstfr/MyRubinLSSTpipelines/blob/main/01_LSSTCamPhotometry/run_bps_pipeline.sh
  
- see my example to inspect official drp-dp2prep here :


  - viewing depp_coadds with matplotlib : https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_LSSTCamDeepCoaddsMosaicMatplotlib.ipynb
    
  - viewing depp_coadds with firefly : https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_LSSTCamDeepCoaddsMosaicFirefly.ipynb
 

#### My Butler/ DP2 actual collections in the butler `dp2_prep`
```python
# The output repo is tagged with the Jira ticket number "DM-40356":
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# bad : crash collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'

# bad : collection = "LSSTComCam/runs/DRP/20241101_20241211/w_2024_51/DM-48233"

# not working perhaps because I am using w_2025_10 version
# bad : no ccd visit collection = "LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864"
# bad : no ccd visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_15/DM-50050'
# bad : no cce visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864'
# bad : no cce visit collection collection = 'LSSTComCam/runs/DRP/DP1/w_2025_13/DM-49751'
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + instrument + "'"
```

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SKYMAP = "lsst_cells_v2"  # adjust to actual skymap used at USDF
INSTRUMENT = "LSSTCam"  # or LSSTComCam / HSC for tests
COLLECTIONS = "LSSTCam/runs/DRP/20250415/w_2025_15/DM-XXXXX"  # ← update
# ──────────────────────────────────────────────────────────────────────────────

if TRACT is not None and PATCH is not None:
    # All visitIds for this diaObject
    visit_list = sorted(df_matched["visitId_int"].dropna().astype(int).unique().tolist())
    visit_str = ", ".join(str(v) for v in visit_list)

    data_query = (
        f"instrument='{INSTRUMENT}' "
        f"AND skymap='{SKYMAP}' "
        f"AND tract={TRACT} "
        f"AND patch={PATCH} "
        f"AND visit IN ({visit_str})"
    )

    print("=" * 70)
    print("dataQuery for bps submit:")
    print("=" * 70)
    print(data_query)
    print("=" * 70)
    print()
    print(f"Number of visits in query : {len(visit_list)}")
    print(f"Visit range               : {visit_list[0]} … {visit_list[-1]}")

    # Optional: write to a text file for copy-paste into the BPS yaml
    query_file = f"dataQuery_diaObj{DIAOBJECT_ID}_tract{TRACT}_patch{PATCH}.txt"
    with open(query_file, "w") as fh:
        fh.write(data_query + "\n")
    print(f"\nQuery saved to: {query_file}")
else:
    print("Cannot generate dataQuery: tract/patch not uniquely determined.")

### 6.1 BPS YAML snippet

In [ ]:
if TRACT is not None and PATCH is not None:
    yaml_snippet = f"""
# ── BPS submit YAML snippet ────────────────────────────────────────────
project: drp_diaObject_{DIAOBJECT_ID}
campaign: tract{TRACT}_patch{PATCH}

pipelineYaml: "$DRP_PIPE_DIR/pipelines/LSSTCam/DRP.yaml"

payload:
  butlerConfig: /repo/main
  inCollection: {COLLECTIONS}
  dataQuery: >
    {data_query}
# ──────────────────────────────────────────────────────────────────────
"""
    print(yaml_snippet)

## 7. Summary

| Item | Value |
|------|-------|
| Butler visits | — |
| consDb visits | — |
| Common visits | — |
| Full tract+patch match | — |
| diaObject tract | — |
| diaObject patch | — |

*Fill in from the cell outputs above.*